# Mean-Variance Portfolio Optimization with cuOpt

<div style="background-color: #e3f2fd; border-left: 6px solid #1976d2; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">📘 Optional Notebook</h4>
We now support solving <strong>Quadratic Programs (QP)</strong>, enabling standard <strong>Markowitz Mean-Variance</strong> portfolio optimization directly on the GPU.

This notebook shows how the same framework extends to QP problems with a quadratic objective with variance as the risk measure <code>w'Σw</code>.
</div>


## Table of Contents

1. [Setup](#setup) — GPU check & imports
2. [Mathematical Formulation](#introduction) — The Markowitz QP
3. [Solve with cuOpt](#solve) — Data, parameters & GPU optimization
4. [Results](#analyze) — Portfolio visualization & statistics
5. [CPU Solver Comparison](#cpu-comparison) — Clarabel vs cuOpt


---

<a id='setup'></a>
## 1. Setup


In [ ]:
# Runs first: CUDA ignores CUDA_VISIBLE_DEVICES once cvxpy, cuOpt, or cuML is imported.
# Set it in ../.env (see .env.sample), GPU 0 when unset; restart the kernel to change it.
from portfolio_optimization.gpu import apply_gpu_selection

print(f"CUDA_VISIBLE_DEVICES={apply_gpu_selection('../.env')!r}")

In [ ]:
!nvidia-smi

In [ ]:
import os

import cvxpy as cp
import numpy as np

from portfolio_optimization import mean_variance_optimizer, utils
from portfolio_optimization.mean_variance_parameters import MeanVarianceParameters
from portfolio_optimization.settings import ApiSettings, ReturnsComputeSettings

---

<a id='introduction'></a>
## 2. Mathematical Formulation

<div style="background-color: #f3e5f5; border-left: 6px solid #9c27b0; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">💡 Mean-Variance (Markowitz) Optimization</h4>
Mean-Variance optimization, introduced by Harry Markowitz in 1952, is the foundation of Modern Portfolio Theory. The objective trades off expected return against portfolio variance:

\begin{align*}
\text{maximize } \quad &\mu^\top w - \lambda_{\text{risk}} \cdot w^\top \Sigma w\\
\text{subject to} \quad 
& \sum_i w_i + c = 1 \quad \text{(self-financing)}\\
& w_i^{\text{min}} \leq w_i \leq w_i^{\text{max}}, \forall i \quad \text{(concentration)}\\
& c^{\text{min}} \leq c \leq c^{\text{max}} \quad \text{(cash)}\\
& L = \|w\|_1 \leq L^{\text{limit}} \quad \text{(leverage)}\\
& T = \|w - w_{\text{prev}}\|_1 \leq T^{\text{limit}} \quad \text{(turnover)}
\end{align*}

This is a **Quadratic Program (QP)** due to the $w^\top \Sigma w$ term. cuOpt 26.02 introduces QP support via a GPU-accelerated barrier solver.
</div>


---

<a id='solve'></a>
## 3. Solve with cuOpt

<div style="background-color: #e8f5e9; border-left: 6px solid #388e3c; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">🚀 End-to-End: Data → Parameters → GPU Solve</h4>
We load S&P 500 price data, compute log-returns, define portfolio constraints, and solve the QP on the GPU using cuOpt.
</div>

<div style="background-color: #fff8e1; border-left: 6px solid #f9a825; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">⚠️ Disclaimer</h4>
Each user is responsible for checking the content of datasets and the applicable licenses and determining if suitable for the intended use.
</div>


In [ ]:
dataset_name = "sp500"
data_path = f"../data/stock_data/{dataset_name}.csv"

if not os.path.exists(data_path):
    utils.download_data(data_path)

returns_dict = utils.calculate_returns(
    data_path,
    {"name": "recent", "range": ("2022-01-01", "2024-01-01")},
    ReturnsComputeSettings(return_type="LOG", freq=1),
)

print(
    f"Assets: {len(returns_dict['tickers'])}  |  "
    f"Observations: {len(returns_dict['returns'])}  |  "
    f"Covariance: {returns_dict['covariance'].shape}"
)

In [ ]:
mean_variance_params = MeanVarianceParameters(
    w_min={"NVDA": 0.1, "others": -0.3},
    w_max={"NVDA": 0.6, "others": 0.4},
    c_min=0.0,
    c_max=0.2,
    L_tar=1.6,
    risk_aversion=1.0,
)

### Solve with CVXPY API

In [ ]:
mean_variance_problem = mean_variance_optimizer.MeanVariance(
    returns_dict=returns_dict,
    mean_variance_params=mean_variance_params,
    api_settings=ApiSettings(api="cvxpy"),
)

cvxpy_gpu_results, cvxpy_gpu_portfolio = (
    mean_variance_problem.solve_optimization_problem(
        solver_settings={"solver": cp.CUOPT, "time_limit": 60, "verbose": True}
    )
)

### Solve using cuOpt Python API

In [ ]:
mean_variance_problem = mean_variance_optimizer.MeanVariance(
    returns_dict=returns_dict,
    mean_variance_params=mean_variance_params,
    api_settings=ApiSettings(api="cuopt_python"),
)

gpu_results, gpu_portfolio = mean_variance_problem.solve_optimization_problem(
    solver_settings={"time_limit": 60, "log_to_console": True}
)

---

<a id='analyze'></a>
## 4. Results


<div style="background-color: #fff8e1; border-left: 6px solid #f9a825; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">📏 Units — every statistic here is daily</h4>
Returns were computed as <strong>1-day log returns</strong> (<code>ReturnsComputeSettings(return_type="LOG", freq=1)</code>), so the expected return, volatility, and variance in the solver printouts above and the statistics below are <strong>daily</strong> quantities, not annualized. Daily log returns annualize additively: a daily expected log return of 0.29% is about a 73% annualized <em>log</em> return, i.e. <code>e^0.73 − 1 ≈ 108%</code> annualized <em>simple</em> return. These are in-sample historical means over the selected regime window — an artifact of the optimizer selecting the sample's best performers, not a return forecast.
</div>


In [ ]:
ax = gpu_portfolio.plot_portfolio(show_plot=True, min_percentage=1)

In [ ]:
weights = gpu_portfolio.weights

expected_return = np.dot(returns_dict["mean"], weights)
portfolio_variance = weights @ returns_dict["covariance"] @ weights
portfolio_std = np.sqrt(portfolio_variance)
sharpe_ratio = expected_return / portfolio_std if portfolio_std > 0 else 0

# Inputs are 1-day LOG returns, so the statistics above are all DAILY figures.
# Log returns annualize additively (x252); volatility scales with sqrt(252).
TRADING_DAYS = 252
annual_log_return = expected_return * TRADING_DAYS
annual_simple_return = np.exp(annual_log_return) - 1
annual_volatility = portfolio_std * np.sqrt(TRADING_DAYS)
annual_sharpe = sharpe_ratio * np.sqrt(TRADING_DAYS)

cash = float(np.asarray(gpu_portfolio.cash).squeeze())

print("Portfolio Statistics (daily, from 1-day log returns):")
print(f"  Expected Return:  {expected_return:.4%} per day")
print(f"  Volatility (Std): {portfolio_std:.4%} per day")
print(f"  Variance:         {portfolio_variance:.6f}")
print(f"  Sharpe Ratio:     {sharpe_ratio:.4f}")
print(f"  Cash Allocation:  {cash:.2%}")
print(f"  Leverage:         {np.sum(np.abs(weights)):.4f}")
print(f"  Number of positions: {np.sum(np.abs(weights) > 0.001)}")
print()
print("Annualized (252 trading days, in-sample):")
print(f"  Log Return:    {annual_log_return:.2%}")
print(f"  Simple Return: {annual_simple_return:.2%}")
print(f"  Volatility:    {annual_volatility:.2%}")
print(f"  Sharpe Ratio:  {annual_sharpe:.2f}")

---

<a id='cpu-comparison'></a>
## 5. CPU Solver Comparison

<div style="background-color: #fff3e0; border-left: 6px solid #f57c00; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">🔄 GPU vs CPU</h4>
Solve the same Mean-Variance problem using <strong>Clarabel</strong> (a CPU-based interior-point solver) and compare the results against the GPU solution above. For small portfolios the GPU advantage in solve time alone may be modest, but running the entire pipeline on GPU — data preprocessing, returns forecasting, scenario generation, and optimization — eliminates costly CPU↔GPU data transfers and delivers end-to-end speedups that grow with problem size.
</div>

<div style="background-color: #fff8e1; border-left: 6px solid #f9a825; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">⏱️ Timing methodology — cold-start solves</h4>
All committed solve times come from a single top-to-bottom execution on a cold kernel, so each solver pays its full one-time cost (CVXPY canonicalization, library initialization). Re-solving the same problem object in a warm session is roughly 40% faster for the CPU solver because CVXPY reuses its cached canonicalization — so interactive re-runs of this cell will report faster CPU times than a fresh end-to-end execution. Cold-start numbers are what a first-time reader reproduces.
</div>

In [ ]:
cpu_problem = mean_variance_optimizer.MeanVariance(
    returns_dict=returns_dict,
    mean_variance_params=mean_variance_params,
    api_settings=ApiSettings(api="cvxpy"),
)

cpu_results, cpu_portfolio = cpu_problem.solve_optimization_problem(
    solver_settings={"solver": cp.CLARABEL}
)

In [ ]:
utils.compare_results(gpu_results.to_dict(), cpu_results.to_dict())

### Compare portfolio composition

`compare_results` above compares solve metrics (solver, time, objective, return, variance). To compare the **holdings**, line the weight vectors up side by side. Because the problem is convex with a single optimum, every solver should agree to solver tolerance (roughly `1e-4` per weight); return and objective agree far tighter.


In [ ]:
import pandas as pd

composition = pd.DataFrame(
    {
        "cuOpt (direct)": np.asarray(gpu_portfolio.weights, dtype=float),
        "cuOpt (via CVXPY)": np.asarray(cvxpy_gpu_portfolio.weights, dtype=float),
        "CLARABEL (CPU)": np.asarray(cpu_portfolio.weights, dtype=float),
    },
    index=returns_dict["tickers"],
)
composition["GPU-CPU diff"] = (
    composition["cuOpt (direct)"] - composition["CLARABEL (CPU)"]
)

top_positions = composition.reindex(
    composition["cuOpt (direct)"].abs().sort_values(ascending=False).index
).head(10)
display(top_positions.round(6))

gpu_cash = float(np.asarray(gpu_portfolio.cash).squeeze())
cpu_cash = float(np.asarray(cpu_portfolio.cash).squeeze())
print(f"max |GPU-CPU| weight difference: {composition['GPU-CPU diff'].abs().max():.2e}")
print(f"L1 weight difference:            {composition['GPU-CPU diff'].abs().sum():.2e}")
print(f"cash — cuOpt: {gpu_cash:.4f} | CLARABEL: {cpu_cash:.4f}")

---

## Summary

<div style="background-color: #e8f5e9; border-left: 6px solid #388e3c; padding: 15px; margin: 15px 0;">
<h4 style="margin-top: 0;">✅ Key Takeaway</h4>
<strong>cuOpt >= 26.02</strong> adds QP support, enabling GPU-accelerated Markowitz Mean-Variance optimization alongside the existing LP-based CVaR solver. This portfolio optimization example handles both problem types — simply choose the appropriate optimizer class.

### Mean-Variance vs CVaR

| Aspect | Mean-Variance | CVaR |
|--------|---------------|------|
| Risk Measure | Variance (symmetric) | Tail risk (asymmetric) |
| Problem Type | QP (Quadratic) | LP (Linear) |
| Data Needed | Mean + Covariance | Scenarios |
| Distribution | Assumes normal | Distribution-free |

</div>
